## Filesystem Pool: 100 Random PyTorch Tensors

A compact end-to-end example that creates 100 random PyTorch tensors, stores them in a `FilesystemPool`, reads a sample back, and verifies integrity.

### Notes
- `FilesystemPool` always mounts under `~/.laila/pools/.mnt/<pool_id>`.
- If that mountpoint is already mounted, the pool reuses it.
- If the image file already exists in the chosen image directory, the pool mounts it.
- If neither exists, the pool creates `<pool_id>.img` in the chosen image directory and mounts it.
- The user only provides the image directory; the image filename and mountpoint are fixed by the pool.


In [ ]:
import os
import tempfile

import torch
import laila

In [ ]:
from laila.pool import FilesystemPool

temp_root = tempfile.mkdtemp(prefix="laila_fs_example_")

filesystem_pool = FilesystemPool(image_dir=temp_root)

laila.add_pool(filesystem_pool, pool_nickname="fs_pool")

print("Image dir :", filesystem_pool.image_dir)
print("Image path:", filesystem_pool.image_path)
print("Mount dir :", filesystem_pool.mount_dir)

In [ ]:
torch.manual_seed(42)

wrapped_tensors = []
for i in range(100):
    tensor = torch.randn(16, 16)
    wrapped = laila.constant(data=tensor, nickname=f"filesystem-random-tensor-{i}")
    wrapped_tensors.append(wrapped)

print("Created tensors:", len(wrapped_tensors))
print("First global_id:", wrapped_tensors[0].global_id)

In [ ]:
futures = []
for wrapped in wrapped_tensors:
    future = laila.memorize(wrapped, pool_nickname="fs_pool")
    futures.append(future)

for future in futures:
    laila.wait(future)

stored_files = sorted(name for name in os.listdir(filesystem_pool.mount_dir) if name.endswith(".json"))
print("Stored tensors:", len(stored_files))
print("First stored file:", stored_files[0])

In [ ]:
sample_indices = [0, 17, 42, 99]

for idx in sample_indices:
    original = wrapped_tensors[idx].data
    future = laila.remember(wrapped_tensors[idx].global_id, pool_nickname="fs_pool")
    recovered = laila.wait(future)
    assert torch.equal(recovered.data, original)

print("Verified sample tensors:", sample_indices)

In [ ]:
for wrapped in wrapped_tensors:
    future = laila.forget(wrapped.global_id, pool_nickname="fs_pool")
    laila.wait(future)

filesystem_pool.close()
print("Cleanup complete")